# Minimal test: is K=1 negative NLL caused by GPU TF32/cancellation?

In [1]:
from pathlib import Path
import json, sys, torch
from torch.utils.data import DataLoader

REPO = Path.cwd().resolve()
if not (REPO / "src").exists():
    REPO = REPO.parent.resolve()
sys.path.insert(0, str(REPO / "src"))

from dalg.data.shard_activations import ActivationBatchDataset, load_meta_index
from dalg.models.mfa import load_mfa

RUN = REPO / "dalg-cache/pile_wikipedia_gemma2b_mfa_100k/q337_k_sweep/layer17_1_337_mfa"
SHARD = REPO / "dalg-cache/pile_gemma2b_activations"

info = json.loads((RUN / "val_indices.json").read_text())
meta = load_meta_index(SHARD, layer=17)
wanted = set(info["val_global_rows"])
val_pos = [i for i, row in enumerate(meta) if row["global_row"] in wanted]

ds = ActivationBatchDataset(
    SHARD, layer=17, row_subset=val_pos,
    batch_size=2048, drop_prefix=32,
    shuffle_shards=False, shuffle_within_shard=False,
    dtype=torch.float16,
)
X = torch.cat(list(DataLoader(ds, batch_size=None, num_workers=0)), dim=0).float()
model = load_mfa(RUN / "mfa_model.pt", map_location="cpu", dtype=torch.float32).eval()
ckpt = torch.load(RUN / "checkpoint.pt", map_location="cpu", weights_only=False)

print("torch", torch.__version__, "cuda build", torch.version.cuda, "cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("X", X.shape, X.dtype)
print("checkpoint best_metric", ckpt["best_metric"])
print("checkpoint last_val_metric", ckpt["last_val_metric"])

torch 2.11.0+cu128 cuda build 12.8 cuda available True
NVIDIA A100-SXM4-40GB
X torch.Size([10080, 2048]) torch.float32
checkpoint best_metric -1025286.573015873
checkpoint last_val_metric -809858.1498015873


In [2]:
@torch.no_grad()
def eval_nll(device, *, precision=None, tf32=None):
    old_precision = torch.get_float32_matmul_precision()
    old_tf32 = torch.backends.cuda.matmul.allow_tf32
    if precision is not None:
        torch.set_float32_matmul_precision(precision)
    if tf32 is not None:
        torch.backends.cuda.matmul.allow_tf32 = tf32
    try:
        m = model.to(device)
        x = X.to(device)
        vals = []
        for i in range(0, x.shape[0], 2048):
            xb = x[i:i+2048]
            vals.append(float(m.nll(xb).cpu()) * xb.shape[0])
        return sum(vals) / x.shape[0]
    finally:
        model.to("cpu")
        torch.set_float32_matmul_precision(old_precision)
        torch.backends.cuda.matmul.allow_tf32 = old_tf32


print("CPU:", eval_nll("cpu"))
if torch.cuda.is_available():
    print("CUDA high, TF32 on:", eval_nll("cuda", precision="high", tf32=True))
    print("CUDA highest, TF32 off:", eval_nll("cuda", precision="highest", tf32=False))
else:
    print("CUDA unavailable in this Python process")

CPU: 6479.670366753472


CUDA high, TF32 on: -1086085.65
CUDA highest, TF32 off: 6491.253357514881


# NLL comparison: layer05 K=1000 vs K=8000 (component-sharded MFA)

Both runs used `split_seed=42`, `val_frac=0.05`, so their val splits are identical (16452 windows). We evaluate mean per-token val NLL for both models on the same 500-window subsample (~112k tokens), in fp32 with TF32 off (required — see the TF32 test above).

For reference, `checkpoint_rank0000.pt` records the final full-val NLL: K=1000 → 1726.06, K=8000 → 1414.74.

In [ ]:
from pathlib import Path
import json, sys, torch
from torch.utils.data import DataLoader

REPO = Path.cwd().resolve()
if not (REPO / "src").exists():
    REPO = REPO.parent.resolve()
sys.path.insert(0, str(REPO / "src"))

from dalg.data.shard_activations import ActivationBatchDataset, load_meta_index
from dalg.models.mfa import load_mfa

SHARD5 = REPO / "dalg-cache/pile_gemma2b_activations"
RUNS = {
    "K=1000": SHARD5 / "layer05_1000_10_component_sharded_mfa",
    "K=8000": SHARD5 / "layer05_8000_10_component_sharded_mfa",
}
N_VAL_WINDOWS = 500  # subsample of the shared val split (~112k tokens)

info5 = json.loads((RUNS["K=1000"] / "val_indices.json").read_text())
meta5 = load_meta_index(SHARD5, layer=5)
wanted5 = set(info5["val_global_rows"])
val_pos5 = [i for i, row in enumerate(meta5) if row["global_row"] in wanted5][:N_VAL_WINDOWS]

ds5 = ActivationBatchDataset(
    SHARD5, layer=5, row_subset=val_pos5,
    batch_size=1024, drop_prefix=32,
    shuffle_shards=False, shuffle_within_shard=False,
    dtype=torch.float16,
)
Xv = torch.cat(list(DataLoader(ds5, batch_size=None, num_workers=0)), dim=0).float()
print("val tokens:", tuple(Xv.shape))

val tokens: (112000, 2048)


In [ ]:
@torch.no_grad()
def mean_val_nll(run_dir, X, device, batch=1024):
    # TF32 corrupts the Mahalanobis term (see the TF32 test above); keep full fp32
    torch.set_float32_matmul_precision("highest")
    torch.backends.cuda.matmul.allow_tf32 = False
    model = load_mfa(run_dir / "mfa_model.pt", map_location="cpu", dtype=torch.float32)
    model = model.eval().to(device)
    x = X.to(device)
    total = 0.0
    for i in range(0, x.shape[0], batch):
        xb = x[i:i + batch]
        total += float(model.nll(xb)) * xb.shape[0]
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return total / x.shape[0]


device = "cuda" if torch.cuda.is_available() else "cpu"
results = {name: mean_val_nll(run, Xv, device) for name, run in RUNS.items()}
for name, v in results.items():
    print(f"{name}: mean val NLL/token = {v:.2f}")
print(f"delta (K=8000 - K=1000) = {results['K=8000'] - results['K=1000']:.2f}")

K=1000: mean val NLL/token = 1707.35
K=8000: mean val NLL/token = 1375.74
delta (K=8000 - K=1000) = -331.61


# Archived K=32000 MFA (old single-file format)

`archived/layer05_32000_mfa` was trained with the older codebase and saved as a single `mfa_model.pt` holding `state_dict` + `meta` (`psi_per_component=False`, i.e. one shared diagonal noise `psi_rho: (D,)`). Current `load_mfa` reads this format directly — the parameter names match and `psi_per_component` is taken from the meta.

Its val split uses the same `split_seed=42` / `val_frac=0.05` and is identical to the K=1000/K=8000 runs, so we reuse `Xv` from the section above. Batch 512 because the `(B, K, q)` intermediates at K=32000 are large.

In [ ]:
RUN32K = SHARD5 / "archived/layer05_32000_mfa"

# smoke-test: current load_mfa on the old single-file format
m32 = load_mfa(RUN32K / "mfa_model.pt", map_location="cpu", dtype=torch.float32)
print("loaded MFA: K =", m32.K, " D =", m32.D, " q =", m32.q,
      " psi_per_component =", m32.psi_per_component)
print("params finite:", all(torch.isfinite(p).all().item() for p in m32.parameters()))

torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32 = False
m32 = m32.eval().to(device)
x = Xv.to(device)
total = 0.0
with torch.no_grad():
    for i in range(0, x.shape[0], 512):
        xb = x[i:i + 512]
        total += float(m32.nll(xb)) * xb.shape[0]
results["K=32000"] = total / x.shape[0]
print(f"K=32000: mean val NLL/token = {results['K=32000']:.2f}")

del m32
if device == "cuda":
    torch.cuda.empty_cache()

loaded MFA: K = 32000  D = 2048  q = 10  psi_per_component = False
params finite: True
K=32000: mean val NLL/token = 1146.92
